## SRP545985

**paper:** [PMID: 39733099](https://pmc.ncbi.nlm.nih.gov/articles/PMC11682125/) - Transcriptomic and genetic profiling in a spontaneous non-human primate model of hypertrophic cardiomyopathy and sudden cardiac death, 2024

**date, curator:** 2025-09-05, Sara Carsanaro

**notes**
* removed WGS and hypertrophic cardiomyopathy libraries (41 libraries)
* tissue site is LVPW = left ventricular posterior wall
    * to capture posterior wall i annotated using EFO:0001656 dorsal
    * left ventricular wall is actually its own uberon term
* did dev stages manually

### annotation summary
run this after annotation is complete

In [26]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,left ventricular posterior wall,UBERON:0036285,wall of left ventricle,perfect match


In [27]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,5.92,MmulDv:0000025,5-year-old stage,perfect match
1,6.83,MmulDv:0000026,6-year-old stage,perfect match
2,6,MmulDv:0000026,6-year-old stage,perfect match
3,3.83,MmulDv:0000023,3-year-old mature stage,perfect match
4,7,MmulDv:0000027,7-year-old stage,perfect match
5,6.92,MmulDv:0000026,6-year-old stage,perfect match
6,7.83,MmulDv:0000027,7-year-old stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP545985"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_92041/3311601719.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 48/48 [00:47<00:00,  1.00it/s]
0 samples 

### library annnotations

In [10]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,,,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,,,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_7,SAMN44695831,5.92,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,45151,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383501,0F14A530D891F5F7C25127BBB5CD7E9B
1,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,,,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,,,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_6,SAMN44695830,6.83,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44639,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383502,6C205098659320C4E226642608420D43
2,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,,,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,,,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_5,SAMN44695829,6,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44824,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383457,FC761ED3907803355B873EE24C7D9E9B
3,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,,,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,,,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_4,SAMN44695828,3.83,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,46484,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383458,3CF483C491912E91E6D19ADD9855A068
4,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,,,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,,,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_3,SAMN44695827,7,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Numb

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [11]:
unique_sorted(library, "infoOrgan")

['left ventricular posterior wall']


In [12]:

# all
library.loc[:,'anatId'] = 'UBERON:0036285'
library.loc[:,'anatName'] = 'wall of left ventricle'
# perfect match, missing child term, other
library.loc[:,'anatAnnotationStatus'] = 'perfect match'
# partial sampling, full sampling, not documented
library.loc[:,'anatBiologicalStatus'] = 'not documented'


# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,UBERON:0036285,wall of left ventricle,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_7,SAMN44695831,5.92,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,45151,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383501,0F14A530D891F5F7C25127BBB5CD7E9B
1,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_6,SAMN44695830,6.83,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44639,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383502,6C205098659320C4E226642608420D43
2,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_5,SAMN44695829,6,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44824,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383457,FC761ED3907803355B873EE24C7D9E9B
3,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,UBERON:0036285,wall of left ventricle,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_4,SAMN44695828,3.83,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,46484,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383458,3CF483C491912E91E6D19ADD9855A068
4,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,nan,,,lv_control_3,SAMN4

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

doing the stage part manually

In [13]:
unique_sorted(library, "infoStage")

['3.83' '5.92' '6' '6.83' '6.92' '7' '7.83']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [14]:
# making these variables because we use them again in the experiment file
my_protocol = 'NEBNext Ultra (II) RNA Library Prep Kit'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,UBERON:0036285,wall of left ventricle,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_7,SAMN44695831,5.92,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,45151,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383501,0F14A530D891F5F7C25127BBB5CD7E9B
1,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_6,SAMN44695830,6.83,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44639,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383502,6C205098659320C4E226642608420D43
2,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_5,SAMN44695829,6,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44824,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383457,FC761ED3907803355B873EE24C7D9E9B
3,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,UBERON:0036285,wall of left ventricle,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_4,SAMN44695828,3.83,year,,,,,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,46484,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383458,3CF483C491912E91E6D19ADD9855A068
4,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_contr

#### globin, replicates

In [15]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [16]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
library.loc[:,'EFOid'] = 'EFO:0001656'
library.loc[:,'EFOname'] = 'dorsal'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,UBERON:0036285,wall of left ventricle,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_7,SAMN44695831,5.92,year,,,EFO:0001656,dorsal,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,45151,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383501,0F14A530D891F5F7C25127BBB5CD7E9B
1,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_6,SAMN44695830,6.83,year,,,EFO:0001656,dorsal,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44639,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383502,6C205098659320C4E226642608420D43
2,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_5,SAMN44695829,6,year,,,EFO:0001656,dorsal,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44824,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383457,FC761ED3907803355B873EE24C7D9E9B
3,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,UBERON:0036285,wall of left ventricle,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_4,SAMN44695828,3.83,year,,,EFO:0001656,dorsal,,,,,05/09/2025,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,46484,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383458,3CF483C491912E91E6D19ADD9855A068
4,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,perfect match,not documented,perfect match,M,,,9544,

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [17]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2025-09-05'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,UBERON:0036285,wall of left ventricle,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_7,SAMN44695831,5.92,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,45151,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383501,0F14A530D891F5F7C25127BBB5CD7E9B
1,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_6,SAMN44695830,6.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44639,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383502,6C205098659320C4E226642608420D43
2,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_5,SAMN44695829,6,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,44824,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383457,FC761ED3907803355B873EE24C7D9E9B
3,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,UBERON:0036285,wall of left ventricle,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_4,SAMN44695828,3.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05,"Total RNA was extracted from the left ventricle posterior wall using the RNeasy Plus Universal Mini Kit (QIAGEN, Hilden, Germany) following manufacturers protocol. Mature mRNA transcripts were selected using the polyA selection method. High quality mature mRNA (RNA Integrity Number; [RIN]&gt;7) were converted to cDNA and and prepared using the NEBNext Ultra II RNA library kit.",,46484,,,,,,TRANSCRIPTOMIC,PolyA,SRR31383458,3CF483C491912E91E6D19ADD9855A068
4,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,perfect match,not documented,perfect mat

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

#### save complete file with correct columns

In [18]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,UBERON:0036285,wall of left ventricle,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_7,SAMN44695831,5.92,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
1,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_6,SAMN44695830,6.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
2,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_5,SAMN44695829,6,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
3,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,UBERON:0036285,wall of left ventricle,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_4,SAMN44695828,3.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
4,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_3,SAMN44695827,7,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
5,SRX26757005,SRP545985,Illumina HiSeq X,SRS23244615,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_2,SAMN44695826,6.92,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
6,SRX26757004,SRP545985,Illumina HiSeq X,SRS23244614,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7.83,perfect match,not documented,perfect match,F,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_1,SAMN44695825,7.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05


### experiment annotations

In [19]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP545985,Transcriptomic and genomics profiling of hypertrophic cardiomyopathy in Rhesus macaques,We conducted RNA-Seq of the left ventricle of five rhesus macaques with hypertrophic cardiomyopathy (HCM) and seven healthy controls for differential gene profiling. We also generated WGS data of 15 severely HCM-affected and 21 healthy macaques.,SRA,,,,NEBNext Ultra (II) RNA Library Prep Kit,full_length,,PRJNA1185182,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [20]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

7

In [21]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
# Bgee 1K
#experiment.loc[:,'projectTags'] = '' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP545985,Transcriptomic and genomics profiling of hypertrophic cardiomyopathy in Rhesus macaques,We conducted RNA-Seq of the left ventricle of five rhesus macaques with hypertrophic cardiomyopathy (HCM) and seven healthy controls for differential gene profiling. We also generated WGS data of 15 severely HCM-affected and 21 healthy macaques.,SRA,partial,,7,NEBNext Ultra (II) RNA Library Prep Kit,full_length,,PRJNA1185182,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [22]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39733099'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11682125/'
experiment.loc[:,'DOI'] = '10.1038/s41598-024-82770-4'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP545985,Transcriptomic and genomics profiling of hypertrophic cardiomyopathy in Rhesus macaques,We conducted RNA-Seq of the left ventricle of five rhesus macaques with hypertrophic cardiomyopathy (HCM) and seven healthy controls for differential gene profiling. We also generated WGS data of 15 severely HCM-affected and 21 healthy macaques.,SRA,partial,,7,NEBNext Ultra (II) RNA Library Prep Kit,full_length,,PRJNA1185182,39733099,https://pmc.ncbi.nlm.nih.gov/articles/PMC11682125/,10.1038/s41598-024-82770-4,,


#### comments

In [23]:
experiment.loc[:,'comment'] = 'removed WGS and hypertrophic cardiomyopathy libraries'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP545985,Transcriptomic and genomics profiling of hypertrophic cardiomyopathy in Rhesus macaques,We conducted RNA-Seq of the left ventricle of five rhesus macaques with hypertrophic cardiomyopathy (HCM) and seven healthy controls for differential gene profiling. We also generated WGS data of 15 severely HCM-affected and 21 healthy macaques.,SRA,partial,,7,NEBNext Ultra (II) RNA Library Prep Kit,full_length,,PRJNA1185182,39733099,https://pmc.ncbi.nlm.nih.gov/articles/PMC11682125/,10.1038/s41598-024-82770-4,,removed WGS and hypertrophic cardiomyopathy libraries


#### save complete file

In [24]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [25]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [28]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [29]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
49734,SRX12905200,SRP344261,NextSeq 550,SRS10847392,CL:0011013,motile sperm cell,BtauDv:0000005,adulthood stage,,motile sperm,Sexually Mature,perfect match,not documented,perfect match,M,Holstein,,9913,NEXTflex Small RNA-Seq Kit v3,full_length,miRNA,,,L04A,SAMN22841582,,,,,,,,low fertility,,SAC,2025-09-05
49735,SRX12905199,SRP344261,NextSeq 550,SRS10847391,CL:0011013,motile sperm cell,BtauDv:0000005,adulthood stage,,motile sperm,Sexually Mature,perfect match,not documented,perfect match,M,Holstein,,9913,NEXTflex Small RNA-Seq Kit v3,full_length,miRNA,,,L02A,SAMN22841581,,,,,,,,low fertility,,SAC,2025-09-05
49736,SRX26756964,SRP545985,Illumina HiSeq X,SRS23244574,UBERON:0036285,wall of left ventricle,MmulDv:0000025,5-year-old stage,,left ventricular posterior wall,5.92,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_7,SAMN44695831,5.92,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
49737,SRX26756963,SRP545985,Illumina HiSeq X,SRS23244573,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_6,SAMN44695830,6.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
49738,SRX26757008,SRP545985,Illumina HiSeq X,SRS23244618,UBERON:0036285,wall of left ventricle,MmulDv:0000026,6-year-old stage,,left ventricular posterior wall,6,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_5,SAMN44695829,6,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
49739,SRX26757007,SRP545985,Illumina HiSeq X,SRS23244617,UBERON:0036285,wall of left ventricle,MmulDv:0000023,3-year-old mature stage,,left ventricular posterior wall,3.83,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_4,SAMN44695828,3.83,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05
49740,SRX26757006,SRP545985,Illumina HiSeq X,SRS23244616,UBERON:0036285,wall of left ventricle,MmulDv:0000027,7-year-old stage,,left ventricular posterior wall,7,perfect match,not documented,perfect match,M,,,9544,NEBNext Ultra (II) RNA Library Prep Kit,full_length,polyA,,,lv_control_3,SAMN44695827,7,year,,,EFO:0001656,dorsal,,,,SAC,2025-09-05


In [30]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
929,SRP435916,Bulk RNAseq of medial preoptic in male juvenil...,Identify novel neuromodulators of juvenile pla...,SRA,total,,16,TruSeq Stranded mRNA,full_length,,PRJNA967274,39052331,https://pmc.ncbi.nlm.nih.gov/articles/PMC11271...,10.1111/gbb.12908,,
930,SRP344261,Characteristics of miRNAs present in bovine sp...,Small non-coding RNAs have been linked to diff...,SRA,total,,13,NEXTflex Small RNA-Seq Kit v3,full_length,,PRJNA777266,35663333,https://pmc.ncbi.nlm.nih.gov/articles/PMC9160602/,10.3389/fendo.2022.874371,38924761[PMID],
931,SRP545985,Transcriptomic and genomics profiling of hyper...,We conducted RNA-Seq of the left ventricle of ...,SRA,partial,,7,NEBNext Ultra (II) RNA Library Prep Kit,full_length,,PRJNA1185182,39733099,https://pmc.ncbi.nlm.nih.gov/articles/PMC11682...,10.1038/s41598-024-82770-4,,removed WGS and hypertrophic cardiomyopathy li...


### add annotations to git

In [31]:
! git pull

Already up to date.


In [32]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [33]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [34]:
! git add $git_experiment_path $git_library_path

In [35]:
! git commit -m $commit_message_exp

[develop 5eec0ac] adding annotated bulk experiment SRP545985
 2 files changed, 8 insertions(+)


In [36]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.92 KiB | 1.92 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git
   28bb6da..5eec0ac  develop -> develop


### add annotation folder and script to git

In [ ]:
! git status

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push